# Tomato Leaf Disease Detector

This notebook trains a convolutional neural network (CNN) to classify tomato leaf diseases.

In [1]:
# Import required libraries for model building and data processing
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, BatchNormalization, GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import os

## Data Preparation

In [2]:
# Configure data augmentation and rescaling
datagen = ImageDataGenerator(
    rotation_range=30,
    horizontal_flip=True,
    vertical_flip=True,
    height_shift_range=0.2,
    width_shift_range=0.2,
    rescale=1./255
)

In [3]:
# Load training and validation datasets
train = datagen.flow_from_directory(r"/kaggle/input/tomatoleaf/tomato/train", batch_size=32, class_mode='categorical', shuffle=True)
val = datagen.flow_from_directory(r"/kaggle/input/tomatoleaf/tomato/val", batch_size=32, class_mode='categorical', shuffle=True)

Found 10000 images belonging to 10 classes.
Found 1000 images belonging to 10 classes.


## Model Building

In [ ]:
# Define the CNN model architecture
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(256, 256, 3)),
    BatchNormalization(),
    MaxPooling2D(2, 2),
    
    Conv2D(64, (3, 3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D(2, 2),
    
    Conv2D(128, (3, 3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D(2, 2),
    
    Conv2D(256, (3, 3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D(2, 2),
    
    GlobalAveragePooling2D(),
    Dense(512, activation='relu'),
    Dropout(0.5),
    Dense(10, activation='softmax')
])

# Compile the model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

## Training

In [ ]:
# Set up callbacks for early stopping and model checkpointing
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
model_checkpoint = ModelCheckpoint('best_model.keras', monitor='val_loss', save_best_only=True)

# Train the model
history = model.fit(train, epochs=50, validation_data=val, callbacks=[early_stopping, model_checkpoint])

## Model Evaluation

In [ ]:
# Load the best saved model
model = tf.keras.models.load_model('best_model.keras')

In [ ]:
# Import libraries for evaluation metrics
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

# Define validation data (to fix NameError in original code)
val = datagen.flow_from_directory(r"/kaggle/input/tomatoleaf/tomato/val", batch_size=32, class_mode='categorical', shuffle=True)

# Set number of batches to evaluate
num_batches = len(val)

# Collect true labels and predictions
y_true_list = []
y_pred_probs_list = []

for i, (x_batch, y_batch) in enumerate(val):
    if i >= num_batches:
        break
    y_true_list.append(y_batch)
    y_pred_probs = model.predict(x_batch)
    y_pred_probs_list.append(y_pred_probs)

# Concatenate lists
y_true = np.concatenate(y_true_list)
y_pred_probs = np.concatenate(y_pred_probs_list)

# Get predicted classes
y_pred = np.argmax(y_pred_probs, axis=1)
y_true_classes = np.argmax(y_true, axis=1)

# Print classification report
print(classification_report(y_true_classes, y_pred))

# Print confusion matrix
print(confusion_matrix(y_true_classes, y_pred))